# 02 Data Exploration

This notebook profiles the extracted MIMIC tables after `mimic.zip` has been unpacked by Notebook 01 into the configured local data directory. It produces reusable schema, missingness, and cohort-diagnostic artifacts for downstream modeling.

## Objectives

- Validate that the extracted tables are readable.
- Build schema previews for the key research tables.
- Estimate table sizes and cohort identifiers.
- Summarize missingness from a configurable sample.
- Profile note category coverage for the clinical text pipeline.
- Save reusable outputs for downstream notebooks and the paper.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas

In [3]:
import pandas as pd

from src.utils.config import load_config
from src.utils.paths import ensure_directories, resolve_project_paths
from src.utils.logging_utils import write_run_manifest
from src.utils.io_utils import save_dataframe_bundle
from src.data_processing.io import list_available_tables, validate_required_tables
from src.data_processing.exploration import build_exploration_bundle, summarize_demographics

In [4]:
local_zip_path = paths['project_root'] / config['dataset']['default_local_zip_path']
available_tables = list_available_tables(paths['extracted_data_dir'])
required_status = validate_required_tables(paths['extracted_data_dir'], config['dataset']['required_tables'])
{
    'configured_local_zip_path': str(local_zip_path),
    'extracted_data_dir': str(paths['extracted_data_dir']),
}, available_tables[:20], required_status

({'configured_local_zip_path': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/mimic.zip',
  'extracted_data_dir': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/data/mimiciii'},
 ['ADMISSIONS.csv',
  'CALLOUT.csv',
  'CAREGIVERS.csv',
  'CHARTEVENTS.csv',
  'CPTEVENTS.csv',
  'DATETIMEEVENTS.csv',
  'DIAGNOSES_ICD.csv',
  'DRGCODES.csv',
  'D_CPT.csv',
  'D_ICD_DIAGNOSES.csv',
  'D_ICD_PROCEDURES.csv',
  'D_ITEMS.csv',
  'D_LABITEMS.csv',
  'ICUSTAYS.csv',
  'INPUTEVENTS_CV.csv',
  'INPUTEVENTS_MV.csv',
  'LABEVENTS.csv',
  'MICROBIOLOGYEVENTS.csv',
  'NOTEEVENTS.csv',
  'OUTPUTEVENTS.csv'],
 {'PATIENTS.csv': True,
  'ADMISSIONS.csv': True,
  'ICUSTAYS.csv': True,
  'CHARTEVENTS.csv': True,
  'LABEVENTS.csv': True,
  'NOTEEVENTS.csv': True,
  'PRESCRIPTIONS.csv': True,
  'MICROBIOLOGYEVENTS.csv': True})

In [5]:
assert all(required_status.values()), (
    f"Some required tables are missing in {paths['extracted_data_dir']}. "
    f"Run 01_dataset_setup first; the local zip is expected at {local_zip_path}."
)

table_names = [
    table_name for table_name in config['exploration']['key_tables']
    if table_name in available_tables
]

table_names

['PATIENTS.csv',
 'ADMISSIONS.csv',
 'ICUSTAYS.csv',
 'NOTEEVENTS.csv',
 'CHARTEVENTS.csv',
 'LABEVENTS.csv',
 'PRESCRIPTIONS.csv',
 'MICROBIOLOGYEVENTS.csv']

## Build exploration artifacts

The profiling functions below read only the amount of data they need. Large tables use chunked row counting, while schema and missingness summaries use small previews or samples.

In [6]:
bundle = build_exploration_bundle(
    extracted_dir=paths['extracted_data_dir'],
    table_names=table_names,
    preview_rows=config['exploration']['preview_rows'],
    sample_rows=config['exploration']['missingness_sample_rows'],
    chunksize=config['dataset']['chunksize'],
    low_memory=config['dataset']['low_memory'],
    id_columns=config['exploration']['cohort_id_columns'],
    note_category_top_k=config['exploration']['note_category_top_k'],
)
demographics_summary = summarize_demographics(
    extracted_dir=paths['extracted_data_dir'],
    low_memory=config['dataset']['low_memory'],
)
bundle.keys(), demographics_summary

(dict_keys(['schema_preview', 'table_summary', 'missingness_sample', 'note_category_summary']),
 {'patient_count': 46520,
  'admission_count': 58976,
  'gender_distribution': {'M': 32950, 'F': 26026},
  'ethnicity_distribution': {'WHITE': 40996,
   'BLACK/AFRICAN AMERICAN': 5440,
   'UNKNOWN/NOT SPECIFIED': 4523,
   'HISPANIC OR LATINO': 1696,
   'OTHER': 1512,
   'ASIAN': 1509,
   'UNABLE TO OBTAIN': 814,
   'PATIENT DECLINED TO ANSWER': 559,
   'ASIAN - CHINESE': 277,
   'HISPANIC/LATINO - PUERTO RICAN': 232}})

In [7]:
artifact_dir = paths['tables_dir'] / '02_data_exploration'
saved_paths = save_dataframe_bundle(bundle, artifact_dir)
saved_paths

{'schema_preview': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/tables/02_data_exploration/schema_preview.csv',
 'table_summary': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/tables/02_data_exploration/table_summary.csv',
 'missingness_sample': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/tables/02_data_exploration/missingness_sample.csv',
 'note_category_summary': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/tables/02_data_exploration/note_category_summary.csv'}

## Preview saved outputs

In [8]:
bundle['table_summary']

,table_name,row_count,column_count,chunk_count,unique_subject_id,unique_hadm_id,unique_icustay_id
0,PATIENTS.csv,46520,8,1,46520,0,0
1,ADMISSIONS.csv,58976,19,1,46520,58976,0
2,ICUSTAYS.csv,61532,12,1,46476,57786,61532
3,NOTEEVENTS.csv,2083180,11,21,46146,58361,0
4,CHARTEVENTS.csv,330712483,15,3308,46467,57272,111689
5,LABEVENTS.csv,27854055,9,279,46252,58151,0
6,PRESCRIPTIONS.csv,4156450,19,42,39363,50216,52151
7,MICROBIOLOGYEVENTS.csv,631726,16,7,39184,48740,0


In [9]:
bundle['schema_preview'].head(20)

,table_name,column_name,dtype_preview,non_null_preview,preview_example
0,PATIENTS.csv,ROW_ID,int64,5,234
1,PATIENTS.csv,SUBJECT_ID,int64,5,249
2,PATIENTS.csv,GENDER,object,5,F
3,PATIENTS.csv,DOB,object,5,2075-03-13 00:00:00
4,PATIENTS.csv,DOD,object,1,nan
5,PATIENTS.csv,DOD_HOSP,object,1,nan
6,PATIENTS.csv,DOD_SSN,float64,0,nan
7,PATIENTS.csv,EXPIRE_FLAG,int64,5,0
8,ADMISSIONS.csv,ROW_ID,int64,5,21
9,ADMISSIONS.csv,SUBJECT_ID,int64,5,22


In [10]:
bundle['missingness_sample'].sort_values(['table_name', 'missing_fraction_sample'], ascending=[True, False]).head(30)

,table_name,column_name,missing_fraction_sample,non_null_count_sample
13,ADMISSIONS.csv,DEATHTIME,0.90044,4978
18,ADMISSIONS.csv,LANGUAGE,0.50502,24749
22,ADMISSIONS.csv,EDREGTIME,0.49676,25162
23,ADMISSIONS.csv,EDOUTTIME,0.49676,25162
20,ADMISSIONS.csv,MARITAL_STATUS,0.19524,40238
19,ADMISSIONS.csv,RELIGION,0.00916,49542
24,ADMISSIONS.csv,DIAGNOSIS,0.00042,49979
8,ADMISSIONS.csv,ROW_ID,0.00000,50000
9,ADMISSIONS.csv,SUBJECT_ID,0.00000,50000
10,ADMISSIONS.csv,HADM_ID,0.00000,50000


In [11]:
bundle['note_category_summary']

,category,note_count,fraction_of_notes,timed_note_fraction,non_empty_text_fraction
0,Nursing/other,822497,0.394828,0.848037,1.0
1,Radiology,522279,0.250712,0.848037,1.0
2,Nursing,223556,0.107315,0.848037,1.0
3,ECG,209051,0.100352,0.848037,1.0
4,Physician,141624,0.067985,0.848037,1.0
5,Discharge summary,59652,0.028635,0.848037,1.0
6,Echo,45794,0.021983,0.848037,1.0
7,Respiratory,31739,0.015236,0.848037,1.0
8,Nutrition,9418,0.004521,0.848037,1.0
9,General,8301,0.003985,0.848037,1.0


In [12]:
pd.DataFrame([
    {'metric': key, 'value': value}
    for key, value in demographics_summary.items()
])

,metric,value
0,patient_count,46520
1,admission_count,58976
2,gender_distribution,"{'M': 32950, 'F': 26026}"
3,ethnicity_distribution,"{'WHITE': 40996, 'BLACK/AFRICAN AMERICAN': 544..."


In [13]:
manifest_path = paths['manifests_dir'] / '02_data_exploration_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='02_data_exploration',
    config=config,
    extra={
        'table_names_profiled': table_names,
        'saved_artifacts': saved_paths,
        'demographics_summary': demographics_summary,
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/manifests/02_data_exploration_manifest.json')

## Research interpretation prompts

Use the generated tables to answer the following before cohort construction:

- Which tables dominate storage and runtime?
- How complete are note timestamps for temporal alignment?
- Which identifiers are available consistently across tables?
- Which variables are so sparse that they may need exclusion or special handling later?
- Are the selected note categories sufficiently represented for multimodal modeling?

## Next notebook

`03_cohort_construction.ipynb` will use these exploration outputs to build the ICU cohort, implement Sepsis-3 labeling, and define leakage-safe patient-level splits.